# Phase 5b — collect resolved environment lock files (reproducibility, Phase 8 task T3)

No GPU needed (CPU runtime is fine, about 1 minute). Reads only `requirements_resolved*.txt` files that earlier runs saved on Drive, and packs them into a small ZIP with no benchmark text, outputs or keys. Upload the ZIP to the assistant; the files become `config/environments/*.lock.txt`.

In [ ]:
# 1 - Mount Drive and list every resolved-requirements file saved by earlier runs
from pathlib import Path
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/norris')
assert ROOT.is_dir(), f'not found: {ROOT}'
found = sorted(p for p in ROOT.rglob('*requirements*resolved*.txt') if p.is_file())
for p in found:
    print(f'{p.stat().st_size:>8,d}  {p.relative_to(ROOT)}')
print(len(found), 'files')

In [ ]:
# 2 - Pack them (text only) with a manifest of SHA-256 hashes, then download
import hashlib, json, zipfile
from datetime import datetime, timezone
from google.colab import files
assert found, 'no requirements_resolved files found; tell the assistant what step 1 printed'
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
zp = Path(f'/content/Norris_env_locks_{stamp}.zip')
manifest = []
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in found:
        rel = str(p.relative_to(ROOT))
        text = p.read_text(errors='replace')
        assert not any(k in text for k in ('sk-', 'hf_')), f'possible secret in {rel}; stop and check'
        z.writestr('locks/' + rel.replace('/', '__'), text)
        manifest.append({'path': rel, 'sha256': hashlib.sha256(p.read_bytes()).hexdigest(), 'lines': text.count('\n')})
    z.writestr('manifest.json', json.dumps({'created_utc': stamp, 'files': manifest}, indent=1))
print(zp, zp.stat().st_size, 'bytes,', len(manifest), 'files')
files.download(str(zp))